# feral — batched multi-RHS solves: a worked example

**The pattern.** Factor a matrix *once*, then solve it against *many* right-hand sides. `feral` shares the expensive supernodal traversal across all columns, so one batched solve is far cheaper than looping a single-RHS solve. Pass `Solver.solve` a 2-D `(n, nrhs)` array and you get back an `(n, nrhs)` solution.

**Where this shows up — any time one factorization meets many vectors:**

- **Parameter sweeps / design exploration** — same physics (one stiffness or conductance matrix), many loads or sources.
- **Sensitivities & gradients** — `jax.jacrev` over a solve, or the columns of `A⁻¹` needed for design derivatives.
- **Uncertainty quantification** — selected entries of `A⁻¹` for variances / leverage scores.
- **Interior-point steps** — predictor and corrector back-solves against one KKT factor (this is what surfaced it in `pounce`).

We make it concrete with a small **steady-state heat-conduction** problem and a sweep over heat-source layouts. Under the hood, for wide `nrhs` `feral` runs each supernode's dense panel as a register-blocked TRSM + GEMM (GitHub issue #57).

In [ ]:
import time
import numpy as np
import scipy.sparse as sp
import feral

rng = np.random.default_rng(0)

## The model: steady-state heat on a square plate

Discretize the steady heat equation $-\nabla^2 u = q$ on an $m \times m$ grid (unit spacing, fixed-temperature boundary) and you get a linear system $L\,u = q$, where $L$ is the 2-D 5-point Laplacian — symmetric positive definite, sparse, the canonical conduction operator. Here $u$ is the temperature field and $q$ the heat-source distribution.

**One plate ⇒ one $L$**, factored once. **Many candidate source layouts ⇒ many right-hand sides** — exactly the batched-solve pattern.

In [ ]:
def laplacian_2d(m):
    """5-point Laplacian on an m x m grid -> (m*m) x (m*m) SPD."""
    n1 = sp.diags([-1.0, 2.0, -1.0], [-1, 0, 1], shape=(m, m))
    eye = sp.identity(m)
    return (sp.kron(eye, n1) + sp.kron(n1, eye)).tocsc()

grid = 40                      # 40 x 40 plate
L_sp = laplacian_2d(grid)
n = L_sp.shape[0]
L = feral.from_scipy(L_sp, symmetric='full')
print(f'plate {grid}x{grid}  ->  n = {n},  nnz(lower) = {L.nnz}')

## Factor once

The factorization is the expensive step; we pay it a single time and reuse it for every source layout below. (The certified inertia confirms $L$ is SPD — all eigenvalues positive.)

In [ ]:
solver = feral.Solver()
status, inertia = solver.factor(L)
print('status :', feral.FactorStatus(status).name)
print('inertia:', inertia)   # SPD -> all positive
assert status == feral.FactorStatus.SUCCESS
assert inertia.n_neg == 0 and inertia.n_zero == 0

## A batch of heat-source layouts

Each column of `Q` is a different heat-source pattern — a localized Gaussian "hot spot" at a random spot on the plate. Solving $L\,U = Q$ returns the steady-state temperature field for **every layout at once**, as the columns of `U`.

In [ ]:
def gaussian_source(cx, cy, width=2.5):
    """A Gaussian hot spot centered at (cx, cy), flattened to length n."""
    yy, xx = np.mgrid[0:grid, 0:grid]
    g = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * width ** 2))
    return g.ravel()

nrhs = 64
centers = rng.integers(4, grid - 4, size=(nrhs, 2))
Q = np.stack([gaussian_source(cx, cy) for cx, cy in centers], axis=1)

U = solver.solve(Q)            # (n, nrhs) temperature fields, ONE batched call
print('Q.shape =', Q.shape, '  U.shape =', U.shape)

## Correctness: batched solve == per-column solves

The batched result must match independent single-RHS solves column by column — the single-RHS path is the trusted reference — and the whole batch must satisfy $L\,U = Q$.

In [ ]:
max_col_diff = 0.0
for j in range(nrhs):
    uj = solver.solve(Q[:, j].copy())
    max_col_diff = max(max_col_diff, np.max(np.abs(U[:, j] - uj)))
print(f'max |batched - single| over all columns = {max_col_diff:.3e}')
assert max_col_diff < 1e-12

batch_res = np.max(np.abs(L_sp @ U - Q))
print(f'max abs residual over batch = {batch_res:.3e}')

## See it: temperature fields for a few layouts

Each solved column reshapes back to the plate. Different source placements give different steady-state temperature distributions — all from the one shared factorization.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, j in zip(axes, range(4)):
    ax.imshow(U[:, j].reshape(grid, grid), cmap='inferno', origin='lower')
    cx, cy = centers[j]
    ax.set_title(f'source at ({cx}, {cy})')
    ax.axis('off')
fig.suptitle('Steady-state temperature for 4 of the 64 source layouts')
plt.tight_layout()
plt.show()

## The payoff: looped single-RHS vs one batched call

Compare the per-RHS cost of looping the single-RHS solve against a single batched call. The batched call amortizes the supernodal traversal and gather/scatter across columns, and for wide `nrhs` runs the dense per-supernode panels as register-blocked TRSM + GEMM.

In [ ]:
def bench(fn, repeat=3):
    best = float('inf')
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best

print(f'{"nrhs":>5}  {"looped":>12}  {"batched":>12}  {"speedup":>8}')
for k in (64, 256):
    Qk = np.stack(
        [gaussian_source(*c) for c in rng.integers(4, grid - 4, size=(k, 2))],
        axis=1,
    )
    cols = [Qk[:, j].copy() for j in range(k)]
    t_loop = bench(lambda: [solver.solve(c) for c in cols])
    t_batch = bench(lambda: solver.solve(Qk))
    per_loop = t_loop / k * 1e6
    per_batch = t_batch / k * 1e6
    print(
        f'{k:>5}  {per_loop:9.2f} us  {per_batch:9.2f} us  '
        f'{per_loop / per_batch:6.2f}x'
    )

At large `nrhs` the batched per-RHS time is a fraction of the looped time. On these 2-D Laplacians the batched solve runs roughly **3–6× faster per RHS** than looping (issue #57 fix #2: row-major working buffers + register-blocked TRSM/GEMM panel kernels). The exact factor depends on problem size and your CPU's SIMD width and cache.

Nothing about the calling code changes — `solver.solve(Q)` with a 2-D `Q` is all you write; `feral` picks the wide-`nrhs` panel kernels automatically.

## Recovering accuracy: refined batched solves

`solve_refined` adds a few steps of **iterative refinement** against the original matrix — cheap insurance that recovers digits on ill-conditioned or near-singular systems (it returns the best iterate, so a refined solve is never worse than the plain one). Pass it a 2-D `B` and the wide refined solve runs through the **same panel kernel** as `solve_many`: one batched solve per refinement step over the still-unconverged columns, instead of looping a single-RHS refined solve per column (GitHub issue #58). Before that fix the refined multi-RHS path bypassed the panel kernel and could be 3–7× slower per RHS than the unrefined batched solve.

Same one-line call — `solver.solve_refined(L, Q)` with a 2-D `Q`:

In [ ]:
Xr = solver.solve_refined(L, Q)        # (n, nrhs), refined AND batched
print('refined batch residual:', np.max(np.abs(L_sp @ Xr - Q)))

# Matches looping the single-RHS refined solve, column by column.
max_col = max(
    np.max(np.abs(Xr[:, j] - solver.solve_refined(L, Q[:, j].copy())))
    for j in range(8)
)
print('max |batched - per-column refined| (first 8 cols):', max_col)

### Does it pay off? Measure it

Time the refined path the same way: looping `solve_refined` per column vs one batched 2-D call.

In [ ]:
print(f'{"nrhs":>5}  {"loop refined":>15}  {"batch refined":>15}  {"speedup":>8}')
for k in (64, 256):
    Qk = np.stack(
        [gaussian_source(*c) for c in rng.integers(4, grid - 4, size=(k, 2))],
        axis=1,
    )
    cols = [Qk[:, j].copy() for j in range(k)]
    t_loop = bench(lambda: [solver.solve_refined(L, c) for c in cols])
    t_batch = bench(lambda: solver.solve_refined(L, Qk))
    print(
        f'{k:>5}  {t_loop / k * 1e6:12.2f} us  {t_batch / k * 1e6:12.2f} us  '
        f'{t_loop / t_batch:6.2f}x'
    )

The batched refined path is roughly **2–3× faster per RHS** than looping the single-RHS refined solve — even on this well-conditioned plate, where refinement does ~0 correction steps and the win is entirely the shared batched **initial** solve. Before issue #58 this path looped the single-RHS refiner and bypassed the panel kernel; it is the **default** for the solver and for pounce's KKT back-solves.

**The nuance — it is not a free lunch.** The batched path amortizes the *solves*; the per-column **residual** `B − A·X` is still computed column by column. On sparse systems that residual is cheap, so the solve dominates and you see the full speedup. On a *dense* Hessian (where the matrix–vector product is as expensive as the solve) the un-batched residual caps the gain — a single-pass batched residual SpMV is the next lever. The speedup also grows with `nrhs` and with how much refinement actually has to do (ill-conditioned / saddle-point KKT systems, where the batched correction solves carry the cost).